# 03 — Advanced LlamaIndex

**Goal**: Explore routing, sub-question query engines, and multi-document queries.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="llama3.2", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("Ready")

## 1. Creating Multiple Document Sources

Let's create separate indices for different topics.

In [ ]:
import os

# Topic-specific documents
topics = {
    "python": ["Python is a high-level programming language known for its readability.",
               "Python supports multiple programming paradigms including OOP and functional."],
    "ml": ["Machine learning uses data to train models that make predictions.",
            "Supervised learning uses labeled data, unsupervised learning finds patterns."],
    "llm": ["Large Language Models are neural networks with billions of parameters.",
             "LLMs are trained on massive text corpora using next-token prediction."]
}

for topic, texts in topics.items():
    os.makedirs(f"./data/{topic}", exist_ok=True)
    for i, text in enumerate(texts):
        with open(f"./data/{topic}/doc_{i}.txt", "w") as f:
            f.write(text)

print("Topic directories created")

In [ ]:
# Build separate indices for each topic
indices = {}
for topic in topics:
    docs = SimpleDirectoryReader(f"./data/{topic}").load_data()
    indices[topic] = VectorStoreIndex.from_documents(docs)
    print(f"Indexed '{topic}' with {len(docs)} docs")

## 2. Query Engine Tools

Each index becomes a "tool" that can be queried.

In [ ]:
tools = []
for topic, index in indices.items():
    engine = index.as_query_engine()
    tool = QueryEngineTool(
        query_engine=engine,
        metadata=ToolMetadata(
            name=f"{topic}_docs",
            description=f"Provides information about {topic.upper()}. "
                        f"Use this tool for questions about {topic}."
        )
    )
    tools.append(tool)

print(f"Created {len(tools)} query engine tools")

## 3. Sub-Question Query Engine

Breaks a complex question into sub-questions, routes each to the right tool, then synthesizes.

In [ ]:
sub_query_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=tools,
    llm=Settings.llm,
    verbose=True
)

response = sub_query_engine.query(
    "What is Python and how does it relate to machine learning?"
)
print(f"\nFinal answer:\n{response}")

## 4. Router Query Engine

Routes a query to the single most relevant tool.

In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=tools,
    verbose=True
)

response = router_engine.query("What are Large Language Models?")
print(f"\nAnswer:\n{response}")

## 5. Comparison: Simple vs. Router vs. Sub-Question

| Approach | When to Use |
|---|---|
| Simple query | Single topic, straightforward question |
| Router | One question → one specific topic |
| Sub-Question | Multi-topic question requiring synthesis |
| Recursive Retrieval | Deep exploration of a topic |

## Key Takeaways

1. **Query Engine Tools** → wrap indices as callable tools
2. **Sub-Question Engine** → decomposes complex questions
3. **Router Engine** → sends question to the right index
4. These patterns let you build sophisticated RAG systems

**Next**: Advanced RAG techniques (HyDE, multi-query, reranking) →